***
# SECTION 0: DATA DESIGN
***

1. Firstly, read the 12 months of New York taxi csv files
    - sourced from: <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>
2. Fetch a New York weather dataset
    - documentation: <https://mesonet.agron.iastate.edu/nws/cf6table.php?station=KNYC&opt=bystation&year=2024>
    - API call: <https://mesonet.agron.iastate.edu/json/cf6.py?station=KNYC&year=2024>
    - originally reported by the *US National Weather Service*
3. Add a hard-coded calendar
    - sourced from: <https://www.officeholidays.com/countries/usa/new-york/2024>
4. Preprocess the data as training, validation & test sets

## **[0.1]** Define dataset

Includes:
1. Reading csv files
2. HTTP-fetching weather data
3. Hard-coded calendar data
4. Final aggregation, by `route`, `time_of_day` and `date`

In [ ]:
seed = 123 # random state seed






import pandas as pd
import requests
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

##############################################################################
#
# Return only one month from `data/` dircectory.
#
# <https://studres.cs.st-andrews.ac.uk/ID5059/Coursework/P2/data/>
# <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>
def read_single_month(month=1):
    path = f"../../data/nyc_taxi_2024-{month:02d}.csv"
    df = pd.read_csv(
        path, 
        parse_dates = ["tpep_pickup_datetime", "tpep_dropoff_datetime"],
        dtype = {"store_and_fwd_flag": str}
    )
    return df

##############################################################################
#
# Read all 12 csvs into 1 dataframe
#
# `nyc_taxi_2024-*.csv` & `taxi_zone_lookup.csv` expected in the `data/` directory
#
# <https://studres.cs.st-andrews.ac.uk/ID5059/Coursework/P2/data/>
# <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>
def raw(month_start=1, month_end=2):
    df = pd.DataFrame() # Empty, base df

    # `month_end+1` because the loop stops 1 before end
    for month in range(month_start, month_end+1):
        tmp = read_single_month(month=month)
        df = pd.concat([df, tmp], ignore_index=True) 

    return df

# Optional: time of day label (useful for report visualisations)
def get_time_of_day(hour):
    if 0 <= hour < 6:
        return "Late Night"
    elif 6 <= hour < 10:
        return "Morning Rush"
    elif 10 <= hour < 16:
        return "Midday"
    elif 16 <= hour < 20:
        return "Evening Rush"
    else:
        return "Evening"

##############################################################################
#
# Transformed version of the dataset, but still **unaggregated**.
#
# 1. cleans dataset for: invalid fares, invalid distances, where pickup is after dropoff
#    and missing pickup items
#
# 2. transforms:
#   a. timeperiod aggregates (what hour, day, week, etc.)
#   b. change `store_and_fwd_flag` from {'Y', 'N'} -> True/False
#   c. real `ratecode` from ID
#   d. real `payment_type` from ID
#   e. real `vendor` from ID
#   f. real `taxi zone` from ID (both pickup zone & dropoff zone)
#   g. real `service zone` from ID (more abstract version of `taxi zone`)
#   h. include `route` taken, i.e. string of `pickup zone` + `dropoff zone`
#   i. same as (h.), but for service zones, as well
#
# 3. drop ID columns
#
# 4. typecast all `object` types -> `category` (expect `store_and_fwd_flag`, which
#    is a boolean)
def clean_single_month(month=1):
    # df = raw(month_start=month_start, month_end=month_end)
    df = read_single_month(month=month)

    # ----------------------------------------------------------------------------#

    ### CLEANING

    # Removing rows with invalid fares (negative or zero fare amounts don't make sense)
    df = df[df["fare_amount"] > 0]

    # Remove rows with invalid trip distances
    df = df[df["trip_distance"] > 0]

    # Remove rows where pickup is after dropoff (data error)
    df = df[df["tpep_pickup_datetime"] < df["tpep_dropoff_datetime"]]

    # Remove any rows with missing pickup times
    df = df.dropna(subset=["tpep_pickup_datetime"])

    # ----------------------------------------------------------------------------#

    ### TRANSFORMATIONS
    #
    # Time aggregates
    df['pickup_date'] = pd.to_datetime(df['tpep_pickup_datetime'].dt.date)
    df['pickup_hr'] = df['tpep_pickup_datetime'].dt.hour
    df['time_of_day'] = df['tpep_pickup_datetime']
    df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
    df['pickup_dow'] = df['tpep_pickup_datetime'].dt.weekday
    df['pickup_week'] = df['tpep_pickup_datetime'].dt.isocalendar().week
    df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
    df['pickup_year'] = df['tpep_pickup_datetime'].dt.year
    df["time_of_day"] = df["tpep_pickup_datetime"].dt.hour.apply(get_time_of_day)

    # # Never used these in the end
    #
    # df['dropoff_hr'] = df['tpep_dropoff_datetime'].dt.hour
    # df['dropoff_day'] = df['tpep_dropoff_datetime'].dt.day
    # df['dropoff_dow'] = df['tpep_dropoff_datetime'].dt.weekday
    # df['dropoff_week'] = df['tpep_dropoff_datetime'].dt.isocalendar().week
    # df['dropoff_month'] = df['tpep_dropoff_datetime'].dt.month
    # df['dropoff_year'] = df['tpep_dropoff_datetime'].dt.year

    # store_and_fwd_flag
    #
    # This flag indicates whether the trip record was held in vehicle memory before
    # sending to the vendor, aka “store and forward,” because the vehicle did not
    # have a connection to the server.
    # Y = store and forward trip
    # N = not a store and forward trip
    #
    # Change flag from 'Y' or 'N' -> True or False
    df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'Y': True, 'N': False})

    # RatecodeID
    #
    # The final rate code in effect at the end of the trip.
    # 1 = Standard rate
    # 2 = JFK
    # 3 = Newark
    # 4 = Nassau or Westchester
    # 5 = Negotiated fare
    # 6 = Group ride
    # 99 = Null/unknown
    # ratecode_mapping = {
    #     1: "Standard Rate",
    #     2: "JFK",
    #     3: "Newark",
    #     4: "Nassua or Westchester",
    #     5: "Negotiated fare",
    #     6: "Group Ride",
    #     99: "Null/unknown",
    # }
    # df['ratecode'] = df['RatecodeID'].map(ratecode_mapping)

    # payment_type
    #
    # A numeric code signifying how the passenger paid for the trip.
    # 0 = Flex Fare trip
    # 1 = Credit card
    # 2 = Cash
    # 3 = No charge
    # 4 = Dispute
    # 5 = Unknown
    # 6 = Voided trip
    # payment_type_mapping = {
    #     0: "Flex Fare trip",
    #     1: "Credit card",
    #     2: "Cash",
    #     3: "No charge",
    #     4: "Dispute",
    #     5: "Unknown",
    #     6: "Voided trip",
    # }
    # df['payment_type'] = df['payment_type'].map(payment_type_mapping)

    # VendorID
    #
    # A code indicating the TPEP provider that provided the record.
    # 1 = Creative Mobile Technologies, LLC
    # 2 = Curb Mobility, LLC
    # 6 = Myle Technologies Inc
    # 7 = Helix
    # vendor_mapping = {
    #     1: "Creative Mobile Technologies, LLC",
    #     2: "Curb Mobility, LLC",
    #     6: "Myle Technologies Inc",
    #     7: "Helix",
    # }
    # df['vendor'] = df['VendorID'].map(vendor_mapping)    

    # Taxi zones
    #
    # .csv file needed to lookup codes
    zones = pd.read_csv("../../data/taxi_zone_lookup.csv")
    manhattan_ids = zones[zones["Borough"] == "Manhattan"]["LocationID"].tolist()

    # Taxi zones: Pickup zone & pickup service zone
    df = df[df["PULocationID"].isin(manhattan_ids)]
    df = df.merge(
        zones[['LocationID', 'Zone', 'service_zone']].rename(columns={
            'Zone': 'pickup_zone',
            # 'service_zone': 'pickup_service_zone',
        }),
        left_on='PULocationID', 
        right_on='LocationID',
        how='left'
    )
    df = df.drop(columns=['LocationID'])

    # Taxi zones: Dropoff zone & dropoff service zone
    df = df.merge(  
        zones[['LocationID', 'Zone', 'service_zone']].rename(columns={
            'Zone': 'dropoff_zone',
            # 'service_zone': 'dropoff_service_zone',
        }),
        left_on='DOLocationID', 
        right_on='LocationID',
        how='left'
    )
    df = df.drop(columns=['LocationID'])

    # Taxi zones: route taken
    df['route'] = df['pickup_zone'].astype(str) + " to " + df['dropoff_zone'].astype(str)
    # df['service_route'] = df['pickup_service_zone'].astype(str) + " to " + df['dropoff_service_zone'].astype(str)

    # ----------------------------------------------------------------------------#

    ### DROP COLUMNS
    columns_to_drop = [
        'RatecodeID',
        'VendorID',
        'PULocationID',
        'DOLocationID',
        'store_and_fwd_flag',
    ]
    df = df.drop(columns=columns_to_drop)


    ### TYPE CAST
    df = df.astype({
        # 'store_and_fwd_flag': 'bool',
        # 'payment_type': 'category',
        # 'ratecode': 'category',
        # 'vendor': 'category',
        'pickup_zone': 'category',
        # 'pickup_service_zone': 'category',
        'dropoff_zone': 'category',
        # 'dropoff_service_zone': 'category',
        'route': 'category',
        # 'service_route': 'category',
    })

    # Only 2024 data
    df = df[df['pickup_year'] == 2024]

    return df


##############################################################################
#
# Documentation:
#     <https://mesonet.agron.iastate.edu/nws/cf6table.php?station=KNYC&opt=bystation&year=2024>
#
# The API specifically calls:
#     <https://mesonet.agron.iastate.edu/json/cf6.py?station=KNYC&year=2024>
#
def read_weather_data():
    # Using new API
    response = requests.get("https://mesonet.agron.iastate.edu/json/cf6.py?station=KNYC&year=2024").json()
    wthr = pd.DataFrame(response['results'])
    wthr['date'] = pd.to_datetime(wthr['valid'])
    wthr = wthr.drop(columns=['name', 'station', 'valid', 'state', 'wfo', 'link', 'product', 'minutes_sunshine', 'possible_sunshine', 'hdd', 'cdd', 'gust_drct', 'avg_drct', 'snowd_12z', 'avg_smph'])

    # CF6 Code	Abbrev	Meaning
    # 1	FG	Fog or Mist
    # 2	DNSEFG	Fog or Vis 0.25 mile or less
    # 3	TS	Thunder
    # 4	IP	Ice pellets
    # 5	GR	Hail
    # 6	FZRA	Freezing Rain or Drizzle
    # 7	DSTSTM	Duststorm or Sandstorm vis 0.25 mile or less
    # 8	HZ	Smoke or Haze
    # 9	BLSN	Blowing Snow
    # X	TOR	Tornado
    # M	M	Missing Data
    wthr['fog']           = wthr['wxcodes'].str.contains('1', na=False).astype(bool)
    wthr['low_vis']       = wthr['wxcodes'].str.contains('2', na=False).astype(bool)
    wthr['thunder']       = wthr['wxcodes'].str.contains('3', na=False).astype(bool)
    wthr['ice']           = wthr['wxcodes'].str.contains('4', na=False).astype(bool)
    wthr['hail']          = wthr['wxcodes'].str.contains('5', na=False).astype(bool)
    wthr['freezing_rain'] = wthr['wxcodes'].str.contains('6', na=False).astype(bool)
    wthr['duststorm']     = wthr['wxcodes'].str.contains('7', na=False).astype(bool)
    wthr['haze']          = wthr['wxcodes'].str.contains('8', na=False).astype(bool)
    wthr['blowing_snow']  = wthr['wxcodes'].str.contains('9', na=False).astype(bool)
    wthr['tornado']       = wthr['wxcodes'].str.contains('X', na=False).astype(bool)

    # A lot of numerical values can include 'T' as an entry
    #
    # This refers to 'trace amounts', i.e. miniscule levels of precipitation; we will just replace with 0
    wthr.replace('T', 0, inplace=True)
    wthr.replace('M', np.nan, inplace=True)
    wthr = wthr.infer_objects(copy=False)
    wthr = wthr.astype({
        'snow': 'float64',
        'precip': 'float64',
    })

    wthr = wthr.drop(columns=[
        'wxcodes', # redundant
        'ice', # empty
        'tornado', # empty
        'blowing_snow', # empty
        'duststorm', # empty
        'avg_temp', # using 'high'
        'dep_temp', # using 'high'
        'low', # using 'high'
        'gust_smph', # using 'max_smph'
    ])

    wthr = wthr.rename(columns={
        'high': 'temp_high',
        'max_smph': 'max_wind_speed',
        'cloud_ss': 'cloud_coverage',
    })

    # impute 3 missing 'max_wind_speed' values
    wthr['max_wind_speed'] = wthr['max_wind_speed'].interpolate(method='linear')

    return wthr

##############################################################################
#
# Source: <https://www.officeholidays.com/countries/usa/new-york/2024>
#
# | Date | Holiday Name | Holiday Type |
# |------|--------------|--------------|
# | 01/01/2024 | New Year's Day | National |
# | 15/01/2024 | Martin Luther King Jr. Day | National |
# | 12/02/2024 | Lincoln's Birthday | Government |
# | 19/02/2024 | Washington's Birthday | Regional |
# | 31/03/2024 | Easter Sunday | Not a holiday |
# | 12/05/2024 | Mother's Day | Not a holiday |
# | 27/05/2024 | Memorial Day | National |
# | 16/06/2024 | Father's Day | Not a holiday |
# | 19/06/2024 | Juneteenth | Regional |
# | 04/07/2024 | Independence Day | National |
# | 02/09/2024 | Labor Day | National |
# | 14/10/2024 | Columbus Day | Regional |
# | 05/11/2024 | Election Day | Government |
# | 11/11/2024 | Veterans Day | Regional |
# | 28/11/2024 | Thanksgiving | National |
# | 25/12/2024 | Christmas Day | National |
def read_calendar_data():
    data = {
        "date": [
            "01/01/2024", "15/01/2024", "12/02/2024", "19/02/2024",
            "31/03/2024", "12/05/2024", "27/05/2024", "16/06/2024",
            "19/06/2024", "04/07/2024", "02/09/2024", "14/10/2024",
            "05/11/2024", "11/11/2024", "28/11/2024", "25/12/2024"
        ],
        "holiday": [
            "New Year's Day", "Martin Luther King Jr. Day", "Lincoln's Birthday",
            "Washington's Birthday", "Easter Sunday", "Mother's Day",
            "Memorial Day", "Father's Day", "Juneteenth", "Independence Day",
            "Labor Day", "Columbus Day", "Election Day", "Veterans Day",
            "Thanksgiving", "Christmas Day"
        ],
        "holiday_type": [
            "National", "National", "Government", "Regional", "Not a holiday",
            "Not a holiday", "National", "Not a holiday", "Regional", "National",
            "National", "Regional", "Government", "Regional", "National", "National"
        ]
    }

    calendar = pd.DataFrame(data)
    calendar["date"] = pd.to_datetime(calendar["date"], format="%d/%m/%Y")
    return calendar

##############################################################################
#
# For many months, clean, aggregated, and append monthly data.
groupings = [
    'pickup_month',
    'pickup_week', 
    # 'pickup_day',
    'pickup_date',
    'pickup_dow',
    # 'pickup_hr', 
    'time_of_day',
    # 'pickup_service_zone', 
    'pickup_zone',
    # 'dropoff_service_zone', 
    'dropoff_zone',
    'route',
    # 'service_route',
    # 'vendor', 
    # 'ratecode', 
    # 'payment_type',
    # 'store_and_fwd_flag'
]
def read_agg(month_start=1, month_end=2, groupings=groupings):
    base_df = pd.DataFrame()

    for month in range(month_start, month_end+1):
        print(f"reading data for month {month}")
        tmp_df = clean_single_month(month)
        # tmp_df = tmp_df[tmp_df['pickup_year'] == 2024]
        agg_df = tmp_df.groupby(groupings, as_index=False, observed=True).agg(
            total_ride_count        = ('tpep_pickup_datetime', 'count'), # count number of rides
            # total_passenger_count   = ('passenger_count', 'sum'),
            # avg_passenger_count     = ('passenger_count', 'mean'),
            # total_trip_distance     = ('trip_distance', 'sum'),
            # avg_trip_distance       = ('trip_distance', 'mean'),
            # total_fare_amount       = ('fare_amount', 'sum'),
            avg_fare_amount         = ('fare_amount', 'mean'),
            # total_extra             = ('extra', 'sum'),
            # avg_extra               = ('extra', 'mean'),
            # total_mta_tax           = ('mta_tax', 'sum'),
            # avg_mta_tax             = ('mta_tax', 'mean'),
            # total_tip_amount        = ('tip_amount', 'sum'),
            # avg_tip_amount          = ('tip_amount', 'mean'),
            # total_tolls_amount      = ('tolls_amount', 'sum'),
            # avg_tolls_amount        = ('tolls_amount', 'mean'),
            # total_impr_surcharge    = ('improvement_surcharge', 'sum'),
            # avg_impr_surcharge      = ('improvement_surcharge', 'mean'),
            # total_revenue           = ('total_amount', 'sum'),
            # avg_revenue             = ('total_amount', 'mean'),
            # total_airport_fee       = ('Airport_fee', 'sum'),
            # avg_airport_fee         = ('Airport_fee', 'mean'),
        )
        base_df = pd.concat([base_df, agg_df], ignore_index=True)

    print("collecting weather data")
    weather = read_weather_data()
    base_df = pd.merge(base_df, weather, left_on='pickup_date', right_on='date', how='left')
    base_df = base_df.drop(columns=['date'])

    print("collecting calendar data")
    cal = read_calendar_data()
    base_df = pd.merge(base_df, cal, left_on='pickup_date', right_on='date', how='left')
    base_df['holiday'] = base_df['holiday'].fillna("None")
    base_df = base_df.drop(columns=['date', 'holiday_type'])

    print("calculating lag demand")
    daily = df.groupby(['pickup_date', 'time_of_day', 'route'], as_index=False)['total_ride_count'].sum()
    daily = daily.sort_values(['route', 'time_of_day', 'pickup_date'])
    daily['lag_demand'] = daily.groupby(['route', 'time_of_day'])['total_ride_count'].shift(1)
    daily[daily['route'] == "Upper East Side North to Upper East Side South"]
    df = df.merge(
        daily[['route', 'pickup_date', 'time_of_day', 'lag_demand']],
        on=['time_of_day', 'route', 'pickup_date'],
        how='left'
    )
    df['lag_demand'] = df['lag_demand'].fillna(0)

    return base_df

In [ ]:
# Read the data in
df = read_agg(month_start=1, month_end=12)
df.info()

## **[0.2]** Preprocessing

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Recast object types -> category (just to be sure)
df = df.astype({
    'pickup_zone': 'category',
    'route': 'category',
    'time_of_day': 'category',
    'hail': 'bool',  
    'freezing_rain': 'bool',  
    'holiday': 'category',
})

# Remove waste columns
df = df.drop(errors='ignore', columns=[
    # 'lag_demand',
    'dropoff_zone',
    'pickup_date',
    'service_route',
    'pickup_service_zone',
    'dropoff_service_zone',
    'pickup_week',
    'pickup_day',  
    'avg_fare_amount',        
    # 'hail',
    # 'freezing_rain',
    # 'haze',
])

# Transformations
df['precip'] = df['total_ride_count'] * df['precip'] # Scale precipitaion with lag demand
df['temp_high'] = df['total_ride_count'] * df['temp_high'] # Scale temperature with lag demand

# These columns only have a few categories
#
# => onehot encoding is okay
columns_to_onehot_encode = [
    'pickup_dow',
    'time_of_day',
]

# These columns have LOADS of categories
#
# => ordinal encoding is needed             <---- not doing this gives 500GB encoded dataset
#``
# (tried TargetEncoder, but still ended up at 22GB - sticking to ordinal)
columns_to_ordinal_encode = [
    'pickup_zone',
    'route',
    'holiday',
]



# Split `y` BEFORE pipeline
y = df['total_ride_count']
X = df.drop(columns=['total_ride_count'])

# Pipeline
#
# Transfroms the categorical columns to onehot-encoded types
pipeline = ColumnTransformer(
    transformers=[
        #                               +--- `sparse_output=True` stops the encoder duplicating data
        #                               |
        ("onehot", OneHotEncoder(sparse_output=True), columns_to_onehot_encode), 
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_ordinal_encode),
    ],
    remainder='passthrough'
)
# pipeline.set_output(transform='pandas') # returning pandas cost too much memory
X_encoded = pipeline.fit_transform(X)

# Train vs Test
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=seed)

# Train vs Validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=seed)

***
# SECTION 1: EXPLORATORY DATA ANALYSIS
***

# **TO DO**
1. explain weather trends (below are all important variables)
    - highest temperate
    - cloud coverage
    - precipiation
    - max wind speed
2. explain time of day
    - Bhaskar's heat map
3. expain large monthly trends
4. explain routes
    - most important routes: Upper East Side (South -> North & North -> South are biggest by far)

## **[1.1]** Weather Trends

## **[1.2]** Time of Day

## **[1.3]** Monthly Trends

## **[1.4]** Pickup zones, drop-off zones, and routes

***
# SECTION 2: RIDE DEMAND
***

A total of 3 regressor models were trained, in order to predict `ride demand`:
| Model | Reason | Package |
| ----- | ------ | ------- |
| Random Forest | used as a foundation, given the easy-to-understand nature of decision trees | sklearn |
| Light Gradient Boosted Machine | heavily optimized for largescale data, and more suitable for stronger predictions | lightgbm |
| MLP Neural Net | comparison to deep learning approach | sklearn |

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Analysis helper function
#
# 1. Reports MAE, MSE, and R squared scores
# 2. Plots Residuals vs. Predictions, and Actual vs. Predictions
def analysis(y_true, y_pred):
    # Mean Absolute Error
    mae = mean_absolute_error(y_true, y_pred)
    print(f"Mean Absolute Error: \t{mae:.4f}")

    # Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)
    print(f"Mean Squared Error: \t{mse:.4f}")

    # R squared
    r2 = r2_score(y_true, y_pred)
    print(f"R squared: \t\t{r2:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Residuals
    residuals = y_true - y_pred
    ax[0].scatter(y_pred, residuals, alpha=.4)
    ax[0].axhline(0, color='black', linestyle='--')
    ax[0].set_xlabel("Predicted")
    ax[0].set_ylabel("Residuals")
    ax[0].set_title("Residuals")

    # Actual vs predicted
    ax[1].scatter(y_true, y_pred, alpha=.4)
    ax[1].set_xlabel("Actual")
    ax[1].set_ylabel("Predicted")
    ax[1].set_title("Actual vs. Predicted")

## **[2.1]** Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Fine-tuned model
rf = RandomForestRegressor(
    n_jobs=1,           # parallel compute tends to crash memory (keep it at 1)
    n_estimators=75,    # need reduced processing
    min_samples_leaf=2, # weird clustering around `250` & `400` predictions without this
    max_depth=25,       # maxdepth=25 is large, but there are a lot of features, and validation/testing remain similar to training
    #
    # criterion='absolute_error'        # very low values being mis-assigned    <-------    took too long to compute
)
rf.fit(X_train, y_train)

# Training results
training_predictions = rf.predict(X_train)
analysis(y_train, training_predictions)

# Validation Results
validation_predictions = rf.predict(X_val)
analysis(y_val, validation_predictions)

# Test Results
test_predictions = rf.predict(X_test)
analysis(y_test, test_predictions)

# Clean up (for memory)
del rf
del training_predictions
del validation_predictions
del test_predictions

## **[2.2]** Light GBM

In [ ]:
from lightgbm import LGBMRegressor

lg = LGBMRegressor(
    random_state=seed,
    learning_rate=0.01,
    n_estimators=1000,
    num_leaves=60, # default=31 <--- using alot more leaves than usual 
    max_depth=-1, # unlimited tree depth (this algorithm is focused on width, more than depth)

    # The type of feature importance to be filled into feature_importances_. 
    # If ‘split’, result contains numbers of times the feature is used in a model. 
    #
    # If ‘gain’, result contains total gains of splits which use the feature.
    # importance_type='split',
    importance_type='gain',
)
lg.fit(X_train, y_train)

# Training results
training_predictions = lg.predict(X_train)
analysis(y_train, training_predictions)

# Validation Results
validation_predictions = lg.predict(X_val)
analysis(y_val, validation_predictions)

# Test Results
test_predictions = lg.predict(X_test)
analysis(y_test, test_predictions)

# Clean up (for memory)
del lg
del training_predictions
del validation_predictions
del test_predictions

## **[2.3]** MLP Neural Net

In [ ]:
from sklearn.neural_network import MLPRegressor

nn = MLPRegressor(
    random_state=seed, 
    activation='relu',
    max_iter=2000,
    tol=0.1 # tolerance
)
nn.fit(X_train, y_train)

# Training results
training_predictions = nn.predict(X_train)
analysis(y_train, training_predictions)

# Validation Results
validation_predictions = nn.predict(X_val)
analysis(y_val, validation_predictions)

# Test Results
test_predictions = nn.predict(X_test)
analysis(y_test, test_predictions)

# Clean up (for memory)
del lg
del training_predictions
del validation_predictions
del test_predictions

***
# SECTION 3: FARE AMOUNT
***

A total of 6 regressor models were trained:

## 3 models for predicting the **fare amount of each ride**:
1. Linear
2. Random Forest
3. MLP Neural Net

## 3 models for predicting the **total revenue**:
4. Linear
5. Random Forest
6. MLP Neural Net

## **[3.1]** Fare Amount: Linear

## **[3.2]** Fare Amount: Random Forest

## **[3.3]** Fare Amount: MLP Neural Net

***
# SECTION 4: REVENUE PREDICTIONS
***

Predicting for the **first quarter of 2025**, we can:
- take the average of previous weather results for January (2022, 2023, 2024)
- use the last revenue day of 2024
- update the calendar for 2025